In [1]:
"""
Phase 3 — Supervised Predictive Modeling & OLS Diagnostics (CO4, 10 marks)
Input: cleaned_tracks.csv (from Phase 1)
Output: figures/residuals.png, figures/qq_plot.png, rf_model.pkl
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

## Step 1: Load cleaned data from Phase 1

In [2]:
df = pd.read_csv("cleaned_tracks.csv")
print("Dataset shape:", df.shape)          # expect (76103, 43)

Dataset shape: (76103, 43)


## Step 2: Build feature matrix X and target y

In [3]:
# Using the _scaled columns (not raw) because:
#  - they were already standardized in Phase 1 using scaler.pkl
#  - Ridge/Lasso are scale-sensitive, so unscaled features would distort them
feature_cols = [
    "danceability_scaled", "energy_scaled", "loudness_scaled", "speechiness_scaled",
    "acousticness_scaled", "instrumentalness_scaled", "liveness_scaled",
    "valence_scaled", "tempo_scaled",
    "energy_valence",   # engineered interaction term from Phase 1
    "explicit",
    "key_1", "key_2", "key_3", "key_4", "key_5", "key_6",
    "key_7", "key_8", "key_9", "key_10", "key_11",   # key_0 dropped (reference category)
    "mode_1",                                         # mode_0 (minor) dropped (reference category)
]

X = df[feature_cols]
y = df["popularity"]

# 80/20 split, fixed random_state for reproducibility across runs/teammates
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("NaN check:", X.isnull().sum().sum())   # must be 0 before modeling

X_train: (60882, 23) | X_test: (15221, 23)
NaN check: 0


## Step 3: Train and compare 4 candidate models

In [4]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1,),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    results.append({
        "model": name,
        "Train_R2": r2_score(y_train, train_pred),
        "Test_R2": r2_score(y_test, test_pred),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    })

results_df = pd.DataFrame(results).sort_values("Test_R2", ascending=False)
print("\nModel comparison:")
print(results_df.to_string(index=False))


Model comparison:
       model  Train_R2  Test_R2  Test_RMSE
RandomForest  0.369746 0.182979  16.044915
       Ridge  0.086269 0.091747  16.917025
      Linear  0.086269 0.091746  16.917041
       Lasso  0.079194 0.084744  16.982127


## Step 4: Regression diagnostics (using Linear model, per spec)

In [5]:
os.makedirs("figures", exist_ok=True)
lr = models["Linear"]
residuals = y_test - lr.predict(X_test)

# Residual plot — checks homoscedasticity (constant variance of errors)
plt.figure(figsize=(8, 6))
plt.scatter(lr.predict(X_test), residuals, alpha=0.3)
plt.axhline(0, color="red")
plt.xlabel("Predicted Popularity")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.savefig("figures/residuals.png")
plt.close()

# Q-Q plot — checks whether residuals are normally distributed
plt.figure(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.savefig("figures/qq_plot.png")
plt.close()

print("\nSaved figures/residuals.png and figures/qq_plot.png")


Saved figures/residuals.png and figures/qq_plot.png


## Step 5: OLS coefficient significance (which features actually matter)

In [6]:
# Works cleanly here because Phase 1 used drop_first=True on the dummies,
# which keeps the design matrix full-rank (no singular-matrix error).
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm.astype(float)).fit()
print("\nOLS summary:")
print(ols_model.summary())


OLS summary:
                            OLS Regression Results                            
Dep. Variable:             popularity   R-squared:                       0.086
Model:                            OLS   Adj. R-squared:                  0.086
Method:                 Least Squares   F-statistic:                     249.8
Date:                Tue, 22 Sep 2026   Prob (F-statistic):               0.00
Time:                        11:31:27   Log-Likelihood:            -2.5915e+05
No. Observations:               60882   AIC:                         5.184e+05
Df Residuals:                   60858   BIC:                         5.186e+05
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const       

## Step 6: Permutation feature importance (on the best model, RF)

In [7]:
rf = models["RandomForest"]
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": perm_result.importances_mean,
}).sort_values("importance", ascending=False)

print("\nPermutation importance:")
print(importance_df.to_string(index=False))


Permutation importance:
                feature  importance
instrumentalness_scaled    0.126197
    acousticness_scaled    0.071111
        loudness_scaled    0.054397
          energy_scaled    0.047204
         valence_scaled    0.041093
    danceability_scaled    0.035759
     speechiness_scaled    0.034436
               explicit    0.012995
           tempo_scaled    0.012189
        liveness_scaled    0.010119
         energy_valence    0.007555
                 mode_1    0.001656
                  key_9    0.000158
                  key_1    0.000131
                 key_11    0.000104
                  key_5    0.000093
                  key_2    0.000063
                  key_7    0.000044
                  key_8    0.000042
                  key_4    0.000028
                  key_6   -0.000004
                 key_10   -0.000007
                  key_3   -0.000043


## Step 7: Save the model for Phase 4 handoff

In [8]:
joblib.dump(rf, "rf_model.pkl")
print("\nSaved rf_model.pkl")


Saved rf_model.pkl
